In [1]:
import os

from openai import AzureOpenAI

from app.config.settings import get_settings

s = get_settings()

In [2]:
azure_client = AzureOpenAI(
    azure_endpoint=s.azure_openai_endpoint,
    api_key=s.azure_openai_api_key,
    api_version=s.azure_openai_api_version,
)

In [10]:
response = azure_client.embeddings.create(
    input=["first phrase", "second phrase", "third phrase"],
    model=s.azure_openai_embedding_model,
)

In [ ]:
for item in response.data:
    length = len(item.embedding)
    print(
        f"data[{item.index}]: length={length}, "
        f"[{item.embedding[0]:.4f}, {item.embedding[1]:.4f}, {item.embedding[2]:.4f}, ... {item.embedding[-1]:.4f}]"
    )

In [ ]:
import asyncpg

conn = await asyncpg.connect(
    host=s.postgres_host,
    port=s.postgres_port,
    database=s.postgres_db,
    user=s.postgres_user,
    password=s.postgres_password,
)
print(await conn.fetchval("SELECT version()"))
await conn.close()

In [12]:
import redis.asyncio as aioredis

redis_client = await aioredis.from_url(
    s.redis_url,
    encoding="utf-8",
    decode_responses=True,
    socket_connect_timeout=5,
    socket_timeout=5,
    health_check_interval=30,
)

In [ ]:
print(f"Ping successful: {await redis_client.ping()}")
await redis_client.aclose()

In [ ]:
print(f"Set value: {await redis_client.set('full_name', 'john doe')}")
print(f"Key exists: {await redis_client.exists('full_name')}")
print(f"Get value: {await redis_client.get('full_name')}")

In [ ]:
test_key = "test_key"
test_value = "test_value"
ttl = 60
await redis_client.set(test_key, test_value, ex=ttl if ttl else None)
await redis_client.get(test_key)

In [ ]:
from app.config.settings import get_settings
from app.memory.session_manager import SessionManager
from app.memory.stores.redis_store import RedisStore

s = get_settings()


redis_store = RedisStore(s.redis_url)
await redis_store.connect()

In [ ]:
import datetime

ttl = 3600

isinstance(ttl, datetime.timedelta)

test_key = "test_key"
test_value = "test_value"
ttl = 60
await redis_store.set(key=test_key, value=test_value, ttl=ttl)
await redis_store.get(key="test_key")

In [ ]:
session_manager_service = SessionManager(redis_store)
await session_manager_service.get_history("user-123")

In [ ]:
user_message = "Hello, how are you?"
assistant_message = "I'm doing well, thank you! How can I assist you today?"

await session_manager_service.append_turn("user-123", user_message, assistant_message)

In [ ]:
user_message = "How many days off do I get?"
assistant_message = "Hmmm... let me check"
await session_manager_service.append_turn("user-123", user_message, assistant_message)

In [ ]:
await session_manager_service.get_history("user-123")

In [ ]:
await session_manager_service.clear_session("user-123")

In [1]:
import os
from pathlib import Path

from app.services.prompt_service import PromptService

project_root = Path(os.getcwd()).resolve()
prompts_dir = Path(project_root, "prompts")
print(f"Looking for prompts in: {prompts_dir}")

prompt_service = PromptService(prompts_dir, "test_hr_system_prompt.j2")

Looking for prompts in: /home/brlamore/src/hr_chatbot/prompts


In [20]:
from app.services.retrieval_service import DocumentChunk

messages = prompt_service.build_messages(
    query="How much vacation do I get?",
    chunks=[
        DocumentChunk(
            id=123,
            content="You have 15 days of vacation per year.",
            source="employee handbook",
            score=0.9,
            metadata={"source": "employee handbook", "relevance_score": 0.9},
        )
    ],
    employee_name="John Doe",
    company_name="Acme Corp",
)

for message in messages:
    print(f"{message['role']}: {message['content'][:600]}...")

system: You are an HR assistant
Today: April 13, 2026
Employee: John DoeCompany: Acme Corp
Knowledge
---
**Source:** employee handbook (relevance: 90%)
You have 15 days of vacation per year.
---
...
user: How much vacation do I get?...
